In [106]:
from pathlib import Path

import kagglehub
import pandas as pd
import pandera.pandas as pa
import numpy as np

In [48]:
PATH_RAW: Path = Path("assets/raw/")

In [49]:
# Download latest version
path = kagglehub.dataset_download("paultimothymooney/stock-market-data")

print("Path to dataset files:", path)

Path to dataset files: /home/marcin/.cache/kagglehub/datasets/paultimothymooney/stock-market-data/versions/74


In [50]:
sp500_path = Path(path + '/stock_market_data/sp500/csv')
sp500_path

PosixPath('/home/marcin/.cache/kagglehub/datasets/paultimothymooney/stock-market-data/versions/74/stock_market_data/sp500/csv')

In [51]:
df_goog = pd.read_csv(sp500_path / "GOOG.csv", parse_dates=['Date'], index_col='Date', usecols=['Date', 'Close'], dayfirst=True)

In [52]:
df_goog.head()

,Close
Date,
2004-08-19,2.499133
2004-08-20,2.697639
2004-08-23,2.724787
2004-08-24,2.611960
2004-08-25,2.640104


In [53]:
df_goog.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4612 entries, 2004-08-19 to 2022-12-12
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   4612 non-null   float64
dtypes: float64(1)
memory usage: 72.1 KB


In [55]:
df_goog['Log Return'] = np.log(df_goog["Close"] / df_goog["Close"].shift(1))

In [56]:
df_goog.head()

,Close,Log Return
Date,,
2004-08-19,2.499133,NaN
2004-08-20,2.697639,0.076433
2004-08-23,2.724787,0.010013
2004-08-24,2.611960,-0.042289
2004-08-25,2.640104,0.010717


In [59]:
df_goog = df_goog.drop(columns=["Close"])

In [12]:
df_goog = df_goog.dropna()

In [13]:
df_goog.head()

,Log Return
Date,
2004-08-20,0.076433
2004-08-23,0.010013
2004-08-24,-0.042289
2004-08-25,0.010717
2004-08-26,0.017859


In [60]:
df_goog.loc['2021'].to_parquet(PATH_RAW / "goog_returns_2021.parquet", index=True)

In [61]:
csv_files = list(sp500_path.glob("*.csv"))  # Wczytanie listy plików spółek w katalogu
print(f"Znaleziono {len(csv_files)} plików dla spółek w indeksie S&P 500.")


Znaleziono 409 plików dla spółek w indeksie S&P 500.


In [68]:
all_returns = []
not_in_2021 = []

for f in csv_files:
    ticker = f.stem  # Nazwa pliku jako ticker
    df = pd.read_csv(f, parse_dates=['Date'], index_col='Date', usecols=['Date', 'Close'], dayfirst=True).sort_index()

    df[ticker] = np.log(df["Close"] / df["Close"].shift(1)) #
    df = df[ticker]

    try:
        all_returns.append(df.loc["2021"])
    except KeyError:
        not_in_2021.append(ticker)




In [69]:
not_in_2021

['TIME']

In [73]:
df_returns = pd.concat(all_returns, axis=1).dropna(how='all')
df_returns.shape

(252, 408)

In [81]:
df_returns


,LBTYA,CAG,FITB,COP,SLB,APD,ESS,CHD,PFE,HOLX,...,KEY,PPG,UAL,CPRT,PHM,D,RF,PEG,FBHS,sp_500
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,0.003709,-0.007474,-0.012409,-0.010558,0.005482,-0.018544,-0.039389,-0.007249,0.000000,0.003974,...,-0.007339,-0.020384,-0.038176,-0.053260,-0.021330,-0.017574,-0.010602,-0.045083,-0.010791,NaN
2021-01-05,0.045042,-0.012020,0.013859,0.055782,0.052363,0.051556,-0.004523,-0.009048,0.010270,0.038894,...,0.015835,0.021216,0.041638,-0.004319,-0.027379,-0.012118,0.018634,0.000179,-0.000826,NaN
2021-01-06,0.030593,0.004769,0.091649,0.039138,0.053862,0.026488,0.040748,0.001514,-0.008642,0.026097,...,0.092835,0.031305,0.003221,-0.019586,-0.005371,0.021145,0.059719,0.020773,0.015459,NaN
2021-01-07,-0.009580,-0.056122,0.038254,0.026983,0.019071,-0.016271,-0.022087,-0.012292,0.005140,-0.000769,...,0.018009,0.012677,0.001836,0.025727,0.039362,-0.021693,0.023490,-0.014335,0.028413,NaN
2021-01-08,-0.014350,0.008548,-0.008304,0.000000,-0.001609,0.001576,0.010271,0.005872,0.001887,0.031809,...,-0.013613,-0.016916,-0.007594,0.010780,-0.022853,0.003284,-0.005109,0.021339,-0.021118,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,0.001419,0.002697,0.014521,0.027980,0.009089,0.009108,0.011144,0.008305,0.008312,0.009544,...,0.015274,0.015915,-0.006484,0.017294,0.013269,0.001163,0.017047,0.007405,0.019373,NaN
2021-12-28,0.011280,0.015440,0.001829,-0.001230,0.009008,0.005840,0.008935,-0.001795,-0.020306,-0.021571,...,0.004322,0.008506,0.015359,0.006835,0.016121,0.006180,-0.001829,0.007961,0.010918,NaN
2021-12-29,-0.009509,-0.001769,0.002737,-0.002739,-0.015395,0.004628,0.007381,0.007954,-0.007440,-0.001597,...,0.004732,0.000117,-0.018730,0.007252,0.024051,0.006525,0.000000,0.005929,0.006775,NaN


In [71]:
df_returns.loc['2021'].isna().sum()

LBTYA    0
CAG      0
FITB     0
COP      0
SLB      0
        ..
PHM      0
D        0
RF       0
PEG      0
FBHS     0
Length: 408, dtype: int64

In [82]:
df_returns["sp_500"] = df_returns.iloc[:, 1:].sum(axis=0)

In [114]:
df_index_2021 = pd.read_csv(PATH_RAW / "INDEX_US_S&P_US_SPX_2021.csv", parse_dates=['Date'], index_col='Date', usecols=['Date', 'Close'], thousands=',').sort_index()

In [115]:
df_index_2021['Log Return'] = np.log(df_index_2021["Close"] / df_index_2021["Close"].shift(1))
df_index_2021 = df_index_2021.drop(columns=["Close"]).dropna(how='all')

In [116]:
df_index_2021.head()

,Log Return
Date,
2021-01-05,0.007058
2021-01-06,0.005694
2021-01-07,0.014738
2021-01-08,0.005477
2021-01-11,-0.006576


In [117]:
df_index_2021.shape

(251, 1)

In [118]:
schema_logs_returns = pa.DataFrameSchema(
    {
        "Log Return": pa.Column(
            float,
            checks=[
                pa.Check.greater_than(-1),
                pa.Check.less_than(1),
            ],
            nullable=False,
            ),
    },
        index =  pa.Index(
            pa.DateTime,
            name="Date",
            checks=[
                pa.Check.in_range("2021-01-01", "2021-12-31"),
                pa.Check(lambda idx: idx.is_monotonic_increasing, element_wise=False),
            ]
        ),
    strict=True,
    coerce=True,
)



In [119]:
schema_logs_returns.validate(df_index_2021)

,Log Return
Date,
2021-01-05,0.007058
2021-01-06,0.005694
2021-01-07,0.014738
2021-01-08,0.005477
2021-01-11,-0.006576
...,...
2021-12-27,0.013744
2021-12-28,-0.001011
2021-12-29,0.001401


In [120]:
df_goog_2021 = pd.read_parquet(PATH_RAW / "goog_returns_2021.parquet")

In [121]:
schema_logs_returns.validate(df_goog_2021)

,Log Return
Date,
2021-01-04,-0.013586
2021-01-05,0.007310
2021-01-06,-0.003239
2021-01-07,0.029504
2021-01-08,0.011106
...,...
2021-12-27,0.006243
2021-12-28,-0.010974
2021-12-29,0.000386


In [122]:
df_index_2021.to_parquet(PATH_RAW / "sp500_returns.parquet", index=True)